# Figure 3 FC-before-IS supplement | Supplementary Fig. S3e-g

This notebook tests whether FC changes precede IS changes across independently trained seeds and summarizes the result with onset, pair-transition and alternative sequence estimators for Extended Data Fig. 3e–g.

In [ ]:
from dataclasses import asdict
import hashlib
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def locate_code_dir():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidate = base if base.name == 'Supplementary_fig_code' else base / 'FC-IS_code' / 'Supplementary_fig_code'
        if (candidate / 'utils' / 'fig3_fc_before_is.py').exists():
            return candidate
    raise FileNotFoundError('Cannot locate FC-IS_code/Supplementary_fig_code.')

CODE_DIR = locate_code_dir()
MLP_ROOT = CODE_DIR.parent / 'MLP'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from utils.fig3_fc_before_is import (
    SuppFig6Config, analyze_sequence_histories, collect_balanced_reference_inputs,
    load_run_history, plot_fig3_fc_before_is_supp, save_results_npz,
    save_run_history, set_seed, train_mlp_fc_is_history,
)


## Analysis configuration

The defaults match the main MLP analysis: a 784-100-100-10 ReLU MLP, small-normal initialization, SGD with learning rate 0.05, five epochs and Pearson definitions for both FC and IS. Hidden layer 2 is the preregistered primary layer because it is the layer used for the main FC-before-IS result. Set `layer_indices=(0, 1)` only as an explicitly reported layer extension.

In [ ]:
CONFIG = SuppFig6Config(
    seeds=tuple(range(15)),
    layer_indices=(1,),
    epochs=5,
    learning_rate=0.05,
    analysis_interval=10,
    samples_per_class=100,
    primary_edge_threshold=0.70,
    primary_onset_fraction=0.25,
    primary_smoothing_sigma=1.0,
)

DATA_ROOT = MLP_ROOT / 'data'
CACHE_DIR = CODE_DIR / 'checkpoints' / 'fig3_FC_before_IS_supp'
RESULTS_DIR = CODE_DIR / 'results' / 'fig3_FC_before_IS_supp'
OUTPUT_DIR = CODE_DIR / 'outputs' / 'fig3_FC_before_IS_supp'
DOWNLOAD_DATA = False
FORCE_RETRAIN = False
DEVICE = None  # None selects CUDA when available.

for directory in (CACHE_DIR, RESULTS_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

asdict(CONFIG)

## Fixed FC reference inputs and deterministic loaders

The same class-balanced set of 1,000 MNIST training images is used to estimate FC at every training step and for every seed. Training batches are independently shuffled with a seed-specific PyTorch generator. This separates training-run variability from FC input-sampling variability.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_dataset = datasets.MNIST(
    root=DATA_ROOT, train=True, transform=transform, download=DOWNLOAD_DATA
)
test_dataset = datasets.MNIST(
    root=DATA_ROOT, train=False, transform=transform, download=DOWNLOAD_DATA
)
fixed_inputs, fixed_labels = collect_balanced_reference_inputs(
    train_dataset,
    samples_per_class=CONFIG.samples_per_class,
    num_classes=CONFIG.num_classes,
)


def make_train_loader(seed: int) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )


test_loader = DataLoader(
    test_dataset, batch_size=256, shuffle=False, num_workers=0
)
print('Fixed FC inputs:', fixed_inputs.shape)

## Independent training runs and online FC/IS measurement

Compact FC/IS histories are cached per seed. The cache signature includes every setting that changes training or raw history extraction; changing only downstream thresholds does not require retraining.

In [ ]:
history_signature = {
    key: value
    for key, value in asdict(CONFIG).items()
    if key in {
        'input_size', 'hidden_dims', 'num_classes', 'activation',
        'init_method', 'dropout_p', 'epochs', 'learning_rate',
        'analysis_interval', 'inference_batch_size',
        'samples_per_class', 'layer_indices',
    }
}
signature_text = json.dumps(history_signature, sort_keys=True)
signature_hash = hashlib.sha256(signature_text.encode('utf-8')).hexdigest()[:12]

histories = []
for seed in CONFIG.seeds:
    cache_path = CACHE_DIR / f'mlp_seed_{seed}_{signature_hash}.npz'
    if cache_path.exists() and not FORCE_RETRAIN:
        history = load_run_history(cache_path)
        print(f'Loaded seed {seed}: {cache_path.name}')
    else:
        history = train_mlp_fc_is_history(
            seed=seed,
            train_loader=make_train_loader(seed),
            fixed_inputs=fixed_inputs,
            config=CONFIG,
            eval_loader=test_loader,
            device=DEVICE,
        )
        save_run_history(history, cache_path)
        print(f'Trained seed {seed}: accuracy={history["final_accuracy"]:.3f}')
    histories.append(history)

[(h['seed'], h['final_accuracy'], h['steps'].shape) for h in histories]

## Retained sequence estimators

The primary statistic is `IS transition time - FC transition time`. Extended Data Fig. 3e–g reports fractional-onset, pair-transition and alternative sequence estimates.

In [ ]:
results = analyze_sequence_histories(histories, CONFIG)
results_path = save_results_npz(
    results, RESULTS_DIR / 'fig3_FC_before_IS_supp_results.npz'
)
print('Saved source data:', results_path)
print('Primary delay by seed:', results['seed_primary_delay'])
print('Final accuracy:', results['final_accuracy'])

## Export Supplementary Fig. S3e-g

In [ ]:
figure, statistics = plot_fig3_fc_before_is_supp(
    results,
    output_dir=OUTPUT_DIR,
    basename='fig3_FC_before_IS_supp',
    export_formats=('svg', 'pdf', 'tiff', 'png'),
    dpi=600,
)
plt.show()
statistics

## Reporting checklist

- e: paired FC/IS onset by independently trained seed.
- f: within-run pair transitions summarized once per seed.
- g: alternative transition estimators.
- Unit pairs are descriptive within runs and are not inferential replicates.